# 문제 3: Tesseract를 활용한 약봉투 이미지 문자 인식(OCR) 실습 — macOS 로컬 버전

**환경**: macOS (Apple Silicon / Intel 자동 감지)  
**소재**: 약봉투 이미지 (한글 + 영어 혼용 텍스트)  
**목표**: pytesseract로 약품명·복약안내·주의사항 텍스트 추출  
**가산점 포인트**:
- `lang='kor+eng'` 옵션으로 한글+영어 혼용 추출
- 5단계 전처리 파이프라인 (Grayscale → Upscale → CLAHE → Adaptive Threshold → Morphology → Median Blur)
- PSM 11 + OEM 1 Tesseract 최적화
- 전체 이미지 배치 처리 + 오류 처리
- 약봉투 특화 키워드 하이라이팅 + 빈도 시각화

## 1. 사전 설치 (터미널에서 한 번만 실행)

```bash
# Homebrew로 Tesseract + 모든 언어팩 설치 (kor 포함)
brew install tesseract tesseract-lang

# Python 라이브러리 설치
pip install pytesseract pillow opencv-python matplotlib
```

> `tesseract-lang`은 한국어를 포함한 100+개 언어팩을 모두 설치합니다.  
> 설치 확인: `tesseract --list-langs` 에 `kor`이 보이면 OK.

## 2. 환경 설정 — 환경 변수 및 라이브러리 임포트

In [ ]:
import os, subprocess, platform

# TESSDATA_PREFIX 동적 탐지 (Apple Silicon / Intel 모두 대응)
# Apple Silicon: /opt/homebrew/share/tessdata/
# Intel       : /usr/local/share/tessdata/
candidate_dirs = ['/opt/homebrew/share', '/usr/local/share']
kor_path = ''
for base in candidate_dirs:
    if not os.path.exists(base):
        continue
    result = subprocess.run(
        ['find', base, '-name', 'kor.traineddata'],
        capture_output=True, text=True
    )
    found = result.stdout.strip().split('\n')[0]
    if found:
        kor_path = found
        break

if not kor_path:
    raise RuntimeError(
        "kor.traineddata 미설치. 터미널에서 다음을 실행:\n"
        "  brew install tesseract tesseract-lang"
    )

os.environ['TESSDATA_PREFIX'] = os.path.dirname(kor_path)
os.environ['TESSERACT_LANG']  = 'kor+eng'        # 한글+영어 혼용
# PSM 11: sparse text — 약봉투처럼 흩어진 텍스트에 최적
# OEM 1:  LSTM only — 한국어 인식 필수 (legacy 엔진은 한국어 미지원)
os.environ['TESSERACT_CONFIG'] = '--psm 11 --oem 1'

print(f"플랫폼          : {platform.machine()} ({platform.system()})")
print(f"TESSDATA_PREFIX  : {os.environ['TESSDATA_PREFIX']}")
print(f"TESSERACT_LANG   : {os.environ['TESSERACT_LANG']}")
print(f"TESSERACT_CONFIG : {os.environ['TESSERACT_CONFIG']}")
print()
print("── 설치된 언어 확인 (kor 포함되어야 함) ──")
!tesseract --list-langs

In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib

# macOS 내장 한글 폰트 사용 (AppleSDGothicNeo는 기본 탑재)
matplotlib.rc('font', family='AppleGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료")

## 3. 이미지 경로 설정 — 로컬 폴더

In [ ]:
# 로컬 이미지 폴더 (본인 경로로 수정)
IMAGE_DIR = os.path.expanduser('~/Documents/MelodyGitHub/공부/TIL/deeplearning/약봉투/')

# 이미지 파일 목록 확인
EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
image_paths = sorted(
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(EXTS)
)

print(f"이미지 폴더      : {IMAGE_DIR}")
print(f"발견된 이미지 수 : {len(image_paths)}장")
for p in image_paths:
    print(' ', os.path.basename(p))

## 4. 기본 OCR — 전처리 없이 원본 이미지 텍스트 추출

In [ ]:
def ocr_basic(image_path: str) -> str:
    """원본 이미지에 대한 기본 OCR (전처리 없음)."""
    img = Image.open(image_path)
    lang   = os.environ['TESSERACT_LANG']
    config = os.environ['TESSERACT_CONFIG']
    text = pytesseract.image_to_string(img, lang=lang, config=config)
    return text


# 첫 번째 이미지로 기본 OCR 시연
sample_path = image_paths[0]

img_display = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 6))
plt.imshow(img_display)
plt.title('원본 이미지')
plt.axis('off')
plt.show()

basic_result = ocr_basic(sample_path)
print("=" * 60)
print(f"[기본 OCR 결과 — lang={os.environ['TESSERACT_LANG']}, 전처리 없음]")
print("=" * 60)
print(basic_result)

## 5. [가산점] 전처리 파이프라인으로 OCR 정확도 향상

약봉투 이미지는 배경 무늬·조명 불균일로 인식률이 낮을 수 있습니다.  
아래 5단계 전처리로 인식률을 높입니다.

| 단계 | 기법 | 목적 |
|------|------|------|
| 1 | 흑백 변환 (Grayscale) | 색상 노이즈 제거 |
| 2 | 크기 업스케일 (×2) | 작은 글자 선명화 |
| 3 | CLAHE (대비 제한 적응형 히스토그램 평탄화) | 조명 불균일 보정, 이진화 품질 향상 |
| 4 | 적응형 이진화 (Adaptive Threshold) | 국소 밝기 기준으로 컬러 배경·조명 불균일 대응 |
| 5 | 모폴로지 클로징 + 미디언 블러 | 끊어진 획 복원 및 점 노이즈 제거 |

In [ ]:
# [가산점] 전처리 파이프라인
def preprocess(image_path: str) -> np.ndarray:
    img = cv2.imread(image_path)

    # 1단계: 흑백 변환
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2단계: 2배 업스케일 — 작은 글자 인식률 향상
    scaled = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # 3단계: CLAHE — 이진화 전 조명 불균일을 보정해 대비를 균일하게 만듦
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    equalized = clahe.apply(scaled)

    # 4단계: 적응형 이진화 — 컬러 배경·조명 불균일 이미지에도 강인
    binary = cv2.adaptiveThreshold(
        equalized, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=31,
        C=10
    )

    # 5단계: 모폴로지 클로징 — 이진화로 끊어진 획을 복원 (한글에 특히 효과적)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    # 미디언 블러 — 점 노이즈 최종 제거
    denoised = cv2.medianBlur(closed, 3)

    return denoised


def ocr_with_preprocessing(image_path: str) -> tuple[np.ndarray, str]:
    processed = preprocess(image_path)
    pil_img = Image.fromarray(processed)
    # [가산점] 환경변수 TESSERACT_LANG + CONFIG 참조 → kor+eng + PSM/OEM 최적화
    lang   = os.environ['TESSERACT_LANG']
    config = os.environ['TESSERACT_CONFIG']
    text = pytesseract.image_to_string(pil_img, lang=lang, config=config)
    return processed, text


print("전처리 함수 정의 완료")
print(f"사용 언어  : {os.environ['TESSERACT_LANG']}")
print(f"OCR 설정   : {os.environ['TESSERACT_CONFIG']}")

## 6. [가산점] 전처리 전/후 비교

In [ ]:
processed_img, preprocessed_result = ocr_with_preprocessing(sample_path)

# 시각적 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB))
axes[0].set_title('원본 이미지', fontsize=14)
axes[0].axis('off')

axes[1].imshow(processed_img, cmap='gray')
axes[1].set_title('전처리 후 (Grayscale → Upscale → CLAHE → Threshold → Morphology)', fontsize=14)
axes[1].axis('off')

plt.suptitle('전처리 전/후 비교', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# OCR 결과 비교 출력
print("=" * 60)
print("[원본 OCR 결과]")
print("=" * 60)
print(basic_result)

print("\n" + "=" * 60)
print("[전처리 후 OCR 결과]")
print("=" * 60)
print(preprocessed_result)

## 7. [가산점] 전체 이미지 배치 처리

In [ ]:
# [가산점] 약봉투 이미지 전체 배치 OCR 처리 (오류 처리 포함)
def batch_ocr(paths: list[str]) -> list[dict]:
    results = []
    for path in paths:
        try:
            _, text = ocr_with_preprocessing(path)
            results.append({'file': os.path.basename(path), 'text': text, 'error': None})
            print(f"  완료: {os.path.basename(path)}")
        except Exception as e:
            results.append({'file': os.path.basename(path), 'text': '', 'error': str(e)})
            print(f"  실패: {os.path.basename(path)} → {e}")
    return results


print("배치 OCR 시작...")
all_results = batch_ocr(image_paths)
success = sum(1 for r in all_results if r['error'] is None)
print(f"\n총 {len(all_results)}장 중 {success}장 성공")

In [ ]:
# 전체 이미지 OCR 결과 출력
for i, result in enumerate(all_results, 1):
    print(f"{'=' * 60}")
    print(f"[이미지 {i}] {result['file']}")
    print(f"{'=' * 60}")
    print(result['text'])
    print()

## 8. [가산점] 약봉투 특화 키워드 하이라이팅

추출된 텍스트에서 약품명·복약 정보·주의사항 관련 키워드를 찾아 강조 표시합니다.

In [ ]:
# [가산점] 약봉투 특화 키워드 하이라이팅
PHARMACY_KEYWORDS = [
    # 복약 관련
    '1정', '2정', '3정', '1캡슐', '1회', '2회', '3회', '1일', '2일', '3일',
    '식전', '식후', '취침전', '공복',
    # 보관 관련
    '밀폐용기', '실온보관', '냉장보관', '차광',
    # 주의 관련
    '주의', '금기', '부작용', '졸음', '음주',
    # 약효 관련
    '진통제', '소염', '항생제', '위장약', '소화',
]


def highlight_keywords(text: str, keywords: list[str]) -> str:
    """발견된 키워드를 [★ ★] 로 감싸 강조 표시."""
    for kw in keywords:
        text = text.replace(kw, f'[★{kw}★]')
    return text


sample_text = all_results[0]['text']
highlighted = highlight_keywords(sample_text, PHARMACY_KEYWORDS)

print("=" * 60)
print(f"[키워드 하이라이팅 결과] {all_results[0]['file']}")
print("=" * 60)
print(highlighted)

In [ ]:
# 전체 이미지에서 발견된 키워드 통계
from collections import Counter

keyword_counts: Counter = Counter()
for result in all_results:
    for kw in PHARMACY_KEYWORDS:
        count = result['text'].count(kw)
        if count > 0:
            keyword_counts[kw] += count

print("=" * 60)
print(f"[전체 {len(all_results)}장에서 발견된 약봉투 키워드 빈도]")
print("=" * 60)
for kw, cnt in keyword_counts.most_common():
    print(f"  {kw:10s}: {cnt}회")

# 키워드 빈도 막대 그래프
if keyword_counts:
    labels, values = zip(*keyword_counts.most_common(10))
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values, color='steelblue')
    plt.title('약봉투 키워드 빈도 Top 10', fontsize=14)
    plt.xlabel('키워드')
    plt.ylabel('등장 횟수')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 9. 최종 요약

| 항목 | 내용 |
|------|------|
| 실행 환경 | macOS 로컬 (Homebrew Tesseract) |
| OCR 엔진 | Tesseract + pytesseract |
| 언어 옵션 | `lang='kor+eng'` (환경변수 `TESSERACT_LANG`으로 관리) |
| OCR 설정 | PSM 11 (sparse text) + OEM 1 (LSTM only) |
| 전처리 단계 | Grayscale → 2× Upscale → CLAHE → Adaptive Threshold → Morphology → Median Blur |
| 처리 이미지 수 | 약봉투 전체 배치 처리 (오류 처리 포함) |
| 창의적 기능 | 약봉투 특화 키워드 하이라이팅 + 빈도 시각화 |

**가산점 적용 사항 (코드 주석 표시: `# [가산점]`)**
1. `lang='kor+eng'` — 환경변수(`TESSERACT_LANG`) 기반 한글+영어 혼용 인식
2. 5단계 전처리 파이프라인 (CLAHE + 적응형 이진화 + 모폴로지)
3. PSM 11 + OEM 1 Tesseract 최적화 (한국어 인식 정확도 극대화)
4. 전체 이미지 배치 처리 (개별 실패 시에도 전체 처리 지속)
5. 약봉투 도메인 특화 키워드 하이라이팅 및 빈도 시각화